In [1]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import rasterio
from tqdm import tqdm
import torchvision.transforms.functional as TF

In [3]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install matplotlib


  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ---------------------------------------- 0.0/104.6 kB ? eta -:--:--
     --- ------------------------------------ 10.2/104.6 kB ? eta -:--:--
     -------------- ---------------------- 41.0/104.6 kB 495.5 kB/s eta 0:00:01
     ------------------------------------ 104.6/104.6 kB 754.9 kB/s eta 0:00:00
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
    --------------------------------------- 0.1/8.1 MB 6.4 MB/s eta 0:00:02
   - -------------------------------------- 0.3/8.1 MB 4.1 MB/s eta 0:00:02
   -- ------------------------------------- 0.4/8.1 MB 3.9 MB/s eta 0:00:02
   --- ------------------------------------ 0.6/8.1 MB 4.4 MB/s eta 0:00:02
   ---- ----------------------------------- 1.0/8.1 MB 4.9 MB/s eta 0:00:02
   ------ --------------------------------- 1.2/8.1 MB 5.7 MB/s eta 0:00:02
   ------- ---------------------------


[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
base_dir2 = 'D:/dataset/AMAZON/'

# Training folder
train_img_dir = os.path.join(base_dir2, 'Training/image')
train_mask_dir = os.path.join(base_dir2, 'Training/label')

# Validation folder
val_img_dir = os.path.join(base_dir2, 'Validation/images')
val_mask_dir = os.path.join(base_dir2, 'Validation/masks')

# Test folder
test_img_dir = os.path.join(base_dir2, 'Test/image')
test_mask_dir = os.path.join(base_dir2, 'Test/mask')

In [25]:
# -------------------------------
# Dataset Loader using rasterio
# -------------------------------
class Sentinel2Dataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_paths = sorted([
            os.path.join(image_dir, fname)
            for fname in os.listdir(image_dir)
            if fname.endswith('.tif')
        ])
        self.mask_paths = sorted([
            os.path.join(mask_dir, fname)
            for fname in os.listdir(mask_dir)
            if fname.endswith('.tif')
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.image_paths[idx]) as src:
            image = src.read([1, 2, 3, 4]).astype(np.float32)  # 4-band RGB+NIR

        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1).astype(np.float32)

        image = image / 255.0
        mask = mask / 255.0

        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask).unsqueeze(0)

        # Resize both to 64x64 to match expected input size
        image = TF.resize(image, [64, 64])
        mask = TF.resize(mask, [64, 64])

        return image, mask

In [26]:
# -------------------------------
# TransUNet-like Architecture
# -------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_channels, bridge_channels, out_channels):
        super(UpBlock, self).__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv = ConvBlock(out_channels + bridge_channels, out_channels)

    def forward(self, x, bridge):
        up = self.up(x)
        out = torch.cat([up, bridge], dim=1)
        return self.conv(out)

class SimpleTransUNet(nn.Module):
    def __init__(self, in_channels=4, num_classes=1):
        super(SimpleTransUNet, self).__init__()
        resnet = models.resnet50(pretrained=True)
        resnet.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.enc2 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.enc3 = resnet.layer2
        self.enc4 = resnet.layer3
        self.enc5 = resnet.layer4

        self.proj = nn.Conv2d(2048, 512, kernel_size=1)

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=512, nhead=8),
            num_layers=2
        )

        self.up4 = UpBlock(2048, 1024, 512)
        self.up3 = UpBlock(512, 512, 256)
        self.up2 = UpBlock(256, 256, 128)
        self.up1 = UpBlock(128, 64, 64)
        self.final = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        enc1 = self.enc1(x)
        enc2 = self.enc2(enc1)
        enc3 = self.enc3(enc2)
        enc4 = self.enc4(enc3)
        enc5 = self.enc5(enc4)

        x_proj = self.proj(enc5)
        b, c, h, w = x_proj.size()
        tokens = x_proj.view(b, c, -1).permute(2, 0, 1)
        transformed = self.transformer(tokens).permute(1, 2, 0).view(b, c, h, w)

        x = self.up4(enc5, enc4)
        x = self.up3(x, enc3)
        x = self.up2(x, enc2)
        x = self.up1(x, enc1)
        return torch.sigmoid(self.final(x))


In [ ]:
# -------------------------------
# Training Function
# -------------------------------
def train_model(model, train_loader, val_loader=None, num_epochs=10, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            images, masks = images.to(device), masks.to(device)
            masks = (masks > 0.5).float()
            preds = model(images)
            preds = nn.functional.interpolate(preds, size=masks.shape[2:], mode='bilinear', align_corners=False)
            loss = criterion(preds, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.4f}")

        if val_loader:
            model.eval()
            with torch.no_grad():
                val_loss = 0
                for images, masks in val_loader:
                    images, masks = images.to(device), masks.to(device)
                    masks = (masks > 0.5).float()
                    preds = model(images)
                    preds = nn.functional.interpolate(preds, size=masks.shape[2:], mode='bilinear', align_corners=False)
                    loss = criterion(preds, masks)
                    val_loss += loss.item()
                print(f"Validation Loss: {val_loss/len(val_loader):.4f}")

In [9]:
pip install rasterio

Note: you may need to restart the kernel to use updated packages.


In [28]:
# -------------------------------
# Run Training
# -------------------------------
train_dataset = Sentinel2Dataset(train_img_dir, train_mask_dir)
val_dataset = Sentinel2Dataset(val_img_dir, val_mask_dir)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

model = SimpleTransUNet(in_channels=4)
train_model(model, train_loader, val_loader, num_epochs=10)

Epoch 1: 100%|██████████| 125/125 [04:16<00:00,  2.05s/it]


Epoch 1 Loss: 0.4615
Validation Loss: 0.3421


Epoch 2: 100%|██████████| 125/125 [03:23<00:00,  1.63s/it]


Epoch 2 Loss: 0.2965
Validation Loss: 0.2617


Epoch 3: 100%|██████████| 125/125 [03:05<00:00,  1.49s/it]


...

Epoch 8 Loss: 0.0979
Validation Loss: 0.0863


Epoch 9: 100%|██████████| 125/125 [03:09<00:00,  1.52s/it]


Epoch 9 Loss: 0.0837
Validation Loss: 0.0748


Epoch 10: 100%|██████████| 125/125 [03:31<00:00,  1.69s/it]


Epoch 10 Loss: 0.0724
Validation Loss: 0.0630
